# Degus genome circos overview — no methylation track

Same as `degus_genome_circos_genetic_epigenetic_overview` but the CpG methylation track is removed and the A/B compartment / GC content tracks are re-spaced outward to fill the freed band.

In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


In [ ]:
import pandas as pd
from pycirclize import Circos
from pycirclize.utils import ColorCycler
import numpy as np
import matplotlib.pyplot as plt
from pycirclize.parser import Gff
ColorCycler.set_cmap("Set3")


In [ ]:
fai = pd.read_csv(
    f"{PROJ_ROOT}/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t",
    header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"]
)


In [ ]:
df_gc = pd.read_csv(
    f"{PROJ_ROOT}/figure/circos-plot/feature-overview/assembly_final.sorted.headerRenamed.chrAssigned.mito.gc_content.bed",
    sep=r"\s+",
    header=None,
    names=["chrom", "start", "end", "gc_frac"]
)


In [ ]:
gff_feat = Gff(f"{PROJ_ROOT}/code/command-line-script/annotation-merging/output/hifiasm-041425-denovoEnhanced_peaks2utr_sorted.agat.gff3")

seqid2genefeatures = gff_feat.get_seqid2features(feature_type=None)


In [ ]:
df_ab = pd.read_csv(
    f"{PROJ_ROOT}/figure/hic-plot/eigenvector_track_dropNA.bed",
    sep=r"\s+",
    skiprows=1,
    # header=None,
    names=["chrom", "start", "end", "ab_comp"]
)


In [ ]:
junction_bed = pd.read_csv(
    f"{PROJ_ROOT}/figure/circos-plot/feature-overview/agp_final_contig2scaffold.bed",
    sep="\t",
    header=0,
    dtype={
        "chr": "string",    # Explicit string type
        "start": "int64",   # 64-bit integer
        "end": "int64"},     # 64-bit integer
    names=["chr", "start", "end"],
    na_values=["."],        # Common NA marker in BED files
    keep_default_na=False   # Prevent unwanted NA conversion
)


In [ ]:
%%time

# 2. Filter for chromosomes 
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]]

# # 3. Set gap degrees (5° after last chromosome)
# gap_degrees = 5
# gap_after = [1] * (len(karyotype) - 1) + [gap_degrees]

chr_sizes = dict(zip(karyotype['chr'], karyotype['length']))
circos = Circos(sectors=chr_sizes, start=30, end=359, space=2, endspace=False)
# circos.text("Octodon degus \n assembly", size=12, r=0)

############ ====> PLOT
# 4. Plot GC track
for i, sector in enumerate(circos.sectors):
    print(sector.name)

    ##### PLOT GENE FEATURES 
    # Plot forward/reverse CDS, rRNA, tRNA tracks
    f_cds_track = sector.add_track((90, 95), r_pad_ratio=0.1) #****
    r_cds_track = sector.add_track((85, 90), r_pad_ratio=0.1) #****
    snrna_track = sector.add_track((80, 85), r_pad_ratio=0.1) #****
    lnrna_track = sector.add_track((75, 80), r_pad_ratio=0.1) #****
    trna_track = sector.add_track((70, 75), r_pad_ratio=0.1) #****
    for feature in seqid2genefeatures[sector.name]:
        if feature.type == "CDS":
            if feature.location.strand == 1:
                f_cds_track.genomic_features(feature, fc="tomato", ec="tomato", lw=0.1) #color="firebrick", alpha=1)  # Darker red fc="tomato") fc="firebrick", ec="firebrick",
            else:
                r_cds_track.genomic_features(feature, fc="dodgerblue", ec="dodgerblue", lw=0.1) #color="skyblue", alpha=1)  # Deeper blue fc="skyblue") fc="royalblue", ec="royalblue",
        elif feature.type == "lnc_RNA":
            lnrna_track.genomic_features(feature, fc="gold",ec="gold",lw=0.1) #color="limegreen",alpha=1) # fc="limegreen",ec="limegreen",
        elif feature.type == "snRNA":
            snrna_track.genomic_features(feature, fc="limegreen",ec="limegreen", lw=0.1) ##color="orange", alpha=1) #fc="orange",ec="orange",
        elif feature.type == "tRNA":
            trna_track.genomic_features(feature,  fc="mediumorchid",ec="mediumorchid", lw=0.1) #color="purple", alpha=1)  # Set lw=0.1 to enphasize small tRNA plot


    ####### AB compartment track
    # subset to this chromosome with dummy data fallback
    sub = df_ab[df_ab["chrom"] == sector.name]
    if len(sub) == 0:
        print(f"Using dummy values for AB compartments on {sector.name}")
        # Create dummy data spanning entire chromosome
        dummy_bins = 10  # Number of dummy bins to create
        x = np.linspace(0, sector.size, dummy_bins)
        y_centered = np.zeros(dummy_bins)  # All zeros
        max_dev = 1.0  # Default range for dummy data
    else:
        # Process real data
        x = (sub["start"] + sub["end"]) / 2
        y = sub["ab_comp"].values
        y_centered = y - y.mean()
        max_dev = np.max(np.abs(y_centered)) if len(y_centered) > 0 else 1.0
    # Create track (same for both real and dummy data)
    ab_track = sector.add_track((70, 60))
    ab_track.axis(fc="none", ec="grey", lw=0.5)
    # Fill above/below zero
    pos = np.where(y_centered > 0, y_centered, 0)
    neg = np.where(y_centered < 0, y_centered, 0)
    ab_track.fill_between(x, pos, 0,
                         vmin=-max_dev, vmax=max_dev,
                         color="pink")
    ab_track.fill_between(x, neg, 0,
                         vmin=-max_dev, vmax=max_dev,
                         color="dodgerblue")
    
    ##### PLOT GC coverage
    # subset to thi chromosome
    sub = df_gc[df_gc["chrom"] == sector.name]
    # midpoint of each window
    x = (sub["start"] + sub["end"]) / 2
    y = sub["gc_frac"].values
    # if you want to center on mean:
    y_centered = y - y.mean()
    # create a radial track for GC
    gc_track = sector.add_track((57, 47)) #****
    gc_track.axis(fc="none", ec="grey", lw=0.5)
    # fill above/below zero
    pos = np.where(y_centered > 0, y_centered, 0)
    neg = np.where(y_centered < 0, y_centered, 0)
    max_dev_gc = np.max(np.abs(y_centered))
    gc_track.fill_between(x.values, pos, 0,
                          vmin=-max_dev_gc, vmax=max_dev_gc,
                          color="red")
    gc_track.fill_between(x.values, neg, 0,
                          vmin=-max_dev_gc, vmax=max_dev_gc,
                          color="blue")

    ## Add tick values on the first sector
    if i==0:
        gc_track.yticks(
            y=[-max_dev_gc, 0, max_dev_gc],
            labels=[f"{-max_dev_gc:.2f}" , 0, f"{max_dev_gc:.2f}"],
            tick_length=1,
            label_size=7,
            line_kws=dict(ec="grey", lw=0.5),
            side='left',
            vmin=-max_dev_gc,
            vmax=max_dev_gc
        )
        ab_track.yticks(
            y=[0, 0.5,1.0],           # Y-values where ticks should appear
            labels=[f"{-max_dev:.2f}","0", f"{max_dev:.2f}"], # Labels for these ticks
            side="left",                # Place ticks on the left side
            tick_length=2,              # Length of the tick lines
            label_size=7,              # Font size for labels
        )
        
    ######## Junction track 
    # Add a new track for junction points
    junction_track = sector.add_track((95, 100))  # Just inside your axis track
    # Plot junction points for this chromosome
    chr_junctions = junction_bed[junction_bed["chr"] == sector.name]
    for _, row in chr_junctions.iterrows():
        # Calculate position in degrees
        posS = row["start"]  # or use midpoint: (row["start"] + row["end"]) / 2
        posE = row["end"]  # or use midpoint: (row["start"] + row["end"]) / 2
        
        # Create a line plot instead of scatter to show junctions
        junction_track.line(
            x=[posS, posE],  # Same x position for start and end
            y=[0,100],    # Vertical line from inner to outer radius
            color="black",
            lw=1,
            arc=True,
            ls=":")

    # (optional) also draw your major/minor ticks as before
    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    interval = 50_000_000  # 50 Mb
    outer.xticks_by_interval(
        interval=interval,
        tick_length=3,
        outer=True,
        show_label=True,
        label_size=8,
        label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb",
        line_kws=dict(ec="grey")
    )

    
    # **Minor ticks every 10 Mb without labels**
    outer.xticks_by_interval(
        interval=10_000_000,     # 10 Mb interval :contentReference[oaicite:1]{index=1}
        tick_length=1,           # shorter tick length :contentReference[oaicite:2]{index=2}
        outer=True,
        show_label=False,        # no labels on minor ticks :contentReference[oaicite:3]{index=3}
        line_kws=dict(ec="black", lw=0.5)
    )

    # chromosome label
    mid = (sector.start + sector.end) / 2
    sector.text(text=sector.name, x=mid, r=115,
                adjust_rotation=True, size=10)

# 5. Render
# circos.plotfig()
# _=circos.plotfig()
fig = circos.plotfig()
ax = fig.axes[0]  # Get the polar axes

# Define label positions (track_top, text, color)
track_labels = [
    (95, "1) Chromosomes", "black"),
    (89.5, "2) Forward CDS", "tomato"),
    (84.5, "3) Reverse CDS", "dodgerblue"),
    (79.5, "4) snRNA", "limegreen"),
    (74.5, "5) lncRNA", "gold"),
    (69.5, "6) tRNA", "mediumorchid"),
    (60.5, "7) A/B\ncompartment", "black"),
    (47.5, "8) GC\ncontent", "black"),
]

# Add each label in the gap space (0° angle)
for track_top, text, color in track_labels:
    ax.text(
        np.deg2rad(-0),       # 0° angle (center of gap)
        track_top + 1.5,     # Place text just above track
        text,
        rotation=0,          # Keep horizontal
        ha='left',
        va='bottom',
        color=color,
        size=7,
        fontweight='bold'
    )

fig.tight_layout()
fig.show()
# CPU times: user 1min 10s, sys: 1.97 s, total: 1min 12s
# Wall time: 1min 12s

In [ ]:
fig.savefig(
    "degus_genome_circos_genetic_overview_noMethylation.png",
    dpi=600,
    bbox_inches="tight",
    transparent=False,
    facecolor="white"
)


In [ ]:
fig.savefig(
    "degus_genome_circos_genetic_overview_noMethylation.svg",
    dpi=600,
    bbox_inches="tight",
    transparent=False,
    facecolor="white"
)


In [ ]:
fig.savefig(
    "degus_genome_circos_genetic_overview_noMethylation.pdf",
    dpi=600,
    bbox_inches="tight",
    transparent=False,
    facecolor="white"
)
